# ⏱️ Day 03a: LLD — Rate Limiter Design

---

## 🎯 What You'll Master Today
- Token Bucket algorithm (full implementation)
- Leaky Bucket
- Fixed Window Counter
- Sliding Window Log
- Sliding Window Counter

**One of the most asked LLD + system design questions.** Know all 5 algorithms.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  RATE LIMITER — WHY AND WHERE                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Why rate limit?                                                 ║
║  • Prevent abuse / DDoS attacks                                  ║
║  • Control cost (API calls to 3rd party)                         ║
║  • Ensure fair usage across users                                ║
║  • Protect downstream services                                   ║
║                                                                   ║
║  Where to place?                                                 ║
║  • Client-side: unreliable (client can bypass)                   ║
║  • API Gateway: best for microservices (Kong, AWS APIGW)         ║
║  • Server-side: middleware in each service                       ║
║  • Dedicated service: centralized rate limiter (Redis-based)     ║
║                                                                   ║
║  Rate limit by:                                                  ║
║  • IP address, User ID, API key                                  ║
║  • Endpoint (different limits per route)                         ║
║  • Headers: X-RateLimit-Remaining, X-RateLimit-Reset            ║
║  • HTTP 429 Too Many Requests (when exceeded)                    ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Token Bucket
# ============================================================

import time

class TokenBucket:
    """
    Tokens are added at a fixed rate.
    Each request consumes one token.
    Allows burst up to bucket capacity.
    
    Used by: AWS, Stripe, GitHub API
    """
    def __init__(self, rate: float, capacity: int):
        self.rate = rate          # tokens added per second
        self.capacity = capacity  # max tokens (burst size)
        self.tokens = capacity    # start full
        self.last_refill = time.time()
    
    def _refill(self):
        now = time.time()
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_refill = now
    
    def allow(self, tokens=1) -> bool:
        self._refill()
        if self.tokens >= tokens:
            self.tokens -= tokens
            return True
        return False

# Demo
tb = TokenBucket(rate=2, capacity=5)  # 2 tokens/sec, burst of 5
print("Token Bucket (rate=2/s, burst=5):")
for i in range(8):
    print(f"  Request {i+1}: {'✅ allowed' if tb.allow() else '❌ rejected'} (tokens: {tb.tokens:.1f})")

print("\n⏳ Wait 2 seconds...")
time.sleep(2)
for i in range(3):
    print(f"  Request {i+9}: {'✅ allowed' if tb.allow() else '❌ rejected'} (tokens: {tb.tokens:.1f})")

In [ ]:
# ============================================================
# 2. Leaky Bucket
# ============================================================

from collections import deque

class LeakyBucket:
    """
    Requests enter a queue (bucket).
    Processed at a constant rate (leak rate).
    Queue full → reject.
    
    Difference from Token Bucket:
    - Token: allows burst, then steady
    - Leaky: ALWAYS steady output rate, no burst
    """
    def __init__(self, capacity: int, leak_rate: float):
        self.capacity = capacity    # queue size
        self.leak_rate = leak_rate  # requests processed per second
        self.queue = deque()
        self.last_leak = time.time()
    
    def _leak(self):
        now = time.time()
        elapsed = now - self.last_leak
        leaked = int(elapsed * self.leak_rate)
        for _ in range(min(leaked, len(self.queue))):
            self.queue.popleft()
        if leaked > 0:
            self.last_leak = now
    
    def allow(self, request_id) -> bool:
        self._leak()
        if len(self.queue) < self.capacity:
            self.queue.append(request_id)
            return True
        return False

print("""
LEAKY BUCKET:
  💧 Requests flow in (variable rate)
  🚿 Processed out at CONSTANT rate
  🪣 If bucket full → reject
  
  Best for: traffic shaping where you need smooth output.
  Example: Network traffic control, Shopify uses it.
""")

In [ ]:
# ============================================================
# 3. Fixed Window Counter
# ============================================================

class FixedWindowCounter:
    """
    Count requests in fixed time windows.
    Simple but has boundary spike problem.
    
    Problem: If limit=10/min
      0:00-0:59 → 0 requests
      0:55-0:59 → 10 requests (allowed)
      1:00-1:05 → 10 requests (allowed — new window)
      → 20 requests in 10 seconds! Double the limit!
    """
    def __init__(self, limit: int, window_sec: int = 60):
        self.limit = limit
        self.window_sec = window_sec
        self.count = 0
        self.window_start = int(time.time())
    
    def allow(self) -> bool:
        now = int(time.time())
        current_window = now - (now % self.window_sec)
        
        if current_window != self.window_start:
            self.window_start = current_window
            self.count = 0
        
        if self.count < self.limit:
            self.count += 1
            return True
        return False

fw = FixedWindowCounter(limit=5, window_sec=10)
print("Fixed Window (5 req / 10 sec):")
for i in range(7):
    print(f"  Request {i+1}: {'✅' if fw.allow() else '❌'}")

In [ ]:
# ============================================================
# 4. Sliding Window Log
# ============================================================

class SlidingWindowLog:
    """
    Store timestamp of each request in a sorted log.
    Count requests in the last N seconds.
    
    Most accurate, but highest memory (stores all timestamps).
    """
    def __init__(self, limit: int, window_sec: int = 60):
        self.limit = limit
        self.window_sec = window_sec
        self.timestamps = deque()  # sorted timestamps
    
    def allow(self) -> bool:
        now = time.time()
        cutoff = now - self.window_sec
        
        # Remove expired timestamps
        while self.timestamps and self.timestamps[0] <= cutoff:
            self.timestamps.popleft()
        
        if len(self.timestamps) < self.limit:
            self.timestamps.append(now)
            return True
        return False

print("Sliding Window Log: stores every timestamp.")
print("  Pro: Most accurate — no boundary spikes")
print("  Con: Memory = O(limit) per user")
print("  With Redis: ZADD + ZRANGEBYSCORE + ZCARD")

In [ ]:
# ============================================================
# 5. Sliding Window Counter (Best Balance)
# ============================================================

class SlidingWindowCounter:
    """
    Hybrid: fixed window granularity + sliding window accuracy.
    Weighted sum of current and previous window.
    
    Memory: O(1) per user (just 2 counters)
    Accuracy: approximate but very good in practice.
    """
    def __init__(self, limit: int, window_sec: int = 60):
        self.limit = limit
        self.window_sec = window_sec
        self.prev_count = 0
        self.curr_count = 0
        self.curr_window = int(time.time()) // window_sec
    
    def allow(self) -> bool:
        now = time.time()
        window = int(now) // self.window_sec
        
        if window != self.curr_window:
            self.prev_count = self.curr_count if window == self.curr_window + 1 else 0
            self.curr_count = 0
            self.curr_window = window
        
        # Position within current window (0.0 to 1.0)
        position = (now % self.window_sec) / self.window_sec
        
        # Weighted estimate
        estimated = self.prev_count * (1 - position) + self.curr_count
        
        if estimated < self.limit:
            self.curr_count += 1
            return True
        return False

print("""
╔════════════════════════════════════════════════════════════════╗
║  ALGORITHM COMPARISON                                         ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  Algorithm         │ Memory     │ Accuracy │ Burst  │ Impl.  ║
║  ──────────────────│────────────│──────────│────────│────────║
║  Token Bucket      │ O(1)       │ Good     │ ✅ Yes │ Easy   ║
║  Leaky Bucket      │ O(bucket)  │ Good     │ ❌ No  │ Easy   ║
║  Fixed Window      │ O(1)       │ Poor *   │ ✅ Yes │ Easy   ║
║  Sliding Log       │ O(limit)   │ Perfect  │ ❌ No  │ Medium ║
║  Sliding Counter   │ O(1)       │ Good     │ ~      │ Medium ║
║                                                                ║
║  * Fixed window can allow 2× limit at window boundary         ║
║                                                                ║
║  💡 Interview recommendation: Token Bucket (most versatile)   ║
║     or Sliding Window Counter (best accuracy/memory trade-off)║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design a rate limiter | Token bucket for APIs. Store per-user state in Redis. Return 429 + headers |
| 2 | Token bucket vs leaky? | Token = burst allowed. Leaky = constant output. Token more common for APIs |
| 3 | Fixed window problem? | Boundary spike: 2× limit in a short period at window boundaries |
| 4 | Distributed rate limiting? | Redis INCR + EXPIRE. Or Lua script for atomicity. Cluster for HA |
| 5 | Rate limit headers? | X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Reset, Retry-After |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Know all 5 algorithms + trade-offs                    │
## │  • Token Bucket = go-to for API rate limiting             │
## │  • Sliding Window Counter = best accuracy/memory          │
## │  • Redis for distributed rate limiting (INCR + EXPIRE)   │
## │  • HTTP 429 + rate limit headers in response              │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Elevator System Design**